# DATA EXTRACTION FROM MIMIC-III FOR MetaLearning4EHRs

This file is for extracting heart-related diseases informations(ICD9 Code start with 42)

In [ ]:
# Import libraries
import pandas as pd
import psycopg2
import os
from pathlib import Path

In [ ]:
# Update connection details to eICU
conn = psycopg2.connect("dbname=eicu user=postgres password=postgres host=localhost port=5432")

# Update the path for data extraction here
export_dir=r"data\eICU_first24h\raw"
Path(export_dir).mkdir(exist_ok=True, parents=True)

## Extract considered icd9 code

icd9_code start with V and E are not considered

In [ ]:
query = """
SELECT *
FROM (
	SELECT 
		SPLIT_PART(icd9code, ',', 1) AS icd9_code,
        diagnosisstring,
	    COUNT(DISTINCT patientunitstayid) AS count
	FROM diagnosis
	WHERE diagnosispriority = 'Primary'
		AND icd9code IS NOT NULL
	    AND icd9code <> ''
		AND diagnosisoffset < 24*60 
		GROUP BY icd9code, diagnosisstring
	) d
WHERE d.icd9_code NOT LIKE 'V%' 
	AND d.icd9_code NOT LIKE 'E%'
ORDER BY count DESC;
"""
df = pd.read_sql_query(query, conn, dtype={'icd9_code': str})
df.to_csv(os.path.join(export_dir, "icd9.csv"),index=False, sep=',')
df.info()

## Extract the icd-9-cm code after the first 24h as phenotyping label

In [ ]:
query = """
SELECT 
	patientunitstayid AS stay_id,
	SPLIT_PART(icd9code, ',', 1) AS icd9_code
FROM diagnosis
WHERE icd9code IS NOT NULL
	AND icd9code <> ''
	AND diagnosisoffset >= 24*60
"""
df = pd.read_sql_query(query, conn, dtype={'icd9_code': str})
df.to_csv(os.path.join(export_dir, "diag_after_24h.csv"),index=False, sep=',')
df.info()

## Demography

Age, Gender, ethnicity<br>
16 < Age < 90, in_hospital_expire_flag = 0

In [ ]:
query = f"""
WITH ranked_rows AS (
    SELECT 
        p.uniquePid AS subject_id, 
        p.patientUnitStayID AS stay_id, 
        p.hospitalID, 
        p.ethnicity, 
        p.gender, 
        p.age, 
        p.unitDischargeOffset, 
        p.unitDischargeOffset AS icustaytime, -- Stay time  
        p.hospitaldischargeoffset AS hospdischtime, 
        d.icd9code,
        ROW_NUMBER() OVER (PARTITION BY p.patientUnitStayID, d.icd9code ORDER BY p.unitDischargeOffset DESC) AS row_num
    FROM patient p 
    LEFT JOIN diagnosis d ON d.patientUnitStayID = p.patientUnitStayID
    WHERE NOT p.age = '> 89'
        AND NOT p.age = ''
        AND p.unitDischargeOffset >= 24*60 -- ICU stay time >= 24h
        AND CAST(p.age AS INT) > 16 AND CAST(p.age AS INT) < 90
        AND d.diagnosisoffset <= 24*60
        AND d.diagnosispriority = 'Primary'
        AND d.icd9code IS NOT NULL
        AND d.icd9code <> ''
)
SELECT 
    subject_id, 
    stay_id, 
    hospitalID, 
    ethnicity, 
    gender, 
    age, 
    unitDischargeOffset, 
    icustaytime, 
    hospdischtime, 
    icd9code
FROM ranked_rows
WHERE row_num = 1;
"""
df = pd.read_sql_query(query, conn)
df.to_csv(os.path.join(export_dir, "demographics.csv"),index=False,sep=',')
df.info()

considered_stay_id = df['stay_id'].tolist()
considered_stay_id_str = ', '.join(map(str, considered_stay_id))

## Vital Signs

In [ ]:
query = f"""
SELECT 
	vital.patientUnitStayID as stay_id, 
	vital.observationOffset,
	vital.heartRate,
	vital.systemicSystolic,
	vital.systemicDiastolic,
	vital.systemicMean,
	vital.respiration,
	vital.saO2,
	vital.temperature
FROM vitalperiodic vital
WHERE vital.observationoffset < 24*60 -- first 24h
	AND vital.observationoffset >= 0
	AND vital.patientUnitStayID in ({considered_stay_id_str})
ORDER BY vital.patientUnitStayID, vital.observationOffset
"""

df = pd.read_sql_query(query,conn)
df.to_csv(os.path.join(export_dir, "vital.csv"),index=False,sep=',')
df.info()

## Laboratory FROM Chartevents


In [ ]:
query = f"""
SELECT 
    lab.patientUnitStayID, 
    lab.labresultoffset, 
    lab.labName, 
    lab.labResult
FROM lab
WHERE lab.labName IN (
    'albumin',
    'anion gap',
    'Hct',
    'Hgb',
    'HCO3',
    'total bilirubin',
    'creatinine',
    'chloride',
    'bedside glucose',
    'glucose'
    'BUN', 
    'PT', 
    'PTT', 
    'sodium', 
    'lactate',
    '-bands'
    'potassium',
    'platelets x 1000',
    'WBC x 1000'
    'PT-INR'
)
	AND lab.labresultoffset < 24*60 -- first 24h
  AND lab.labresultoffset >= 0
  AND lab.patientUnitStayID in ({considered_stay_id_str})
ORDER BY lab.patientUnitStayID, lab.labResultOffset;
  """

df = pd.read_sql_query(query,conn)
df.to_csv(os.path.join(export_dir, "labs.csv"),index=False,sep=',')
df.info()

## Extract labels
Including ICU mortality, Remain ICU Length of Stay, and medications after 24h.

In [ ]:
# ICU mortality

query = f"""
SELECT 
    patientunitstayid AS stay_id,
    CASE 
        WHEN hospitaldischargestatus = 'Expired' THEN 1
        WHEN hospitaldischargestatus = 'Alive' THEN 0
        ELSE NULL -- 如果存在其他情况，可以设置为 NULL 或其他值
    END AS icu_mortality
FROM patient
WHERE unitdischargeoffset >= 24*60
  AND patientUnitStayID in ({considered_stay_id_str})
  """

df = pd.read_sql_query(query,conn)
df.to_csv(os.path.join(export_dir, "label_icumortality.csv"),index=False,sep=',')
df.info()

In [ ]:
# ICU Remaining LoS

query = f"""
SELECT 
    patientunitstayid AS stay_id,
    CASE
        WHEN unitdischargeoffset - 24*60 > 0 AND unitdischargeoffset - 24*60 < 24*60 THEN 0 -- Less than 1 day
        WHEN unitdischargeoffset - 24*60 >= 24*60 AND unitdischargeoffset - 24*60 < 2*24*60 THEN 1 -- 1 day to 2 days
        WHEN unitdischargeoffset - 24*60 >= 2*24*60 AND unitdischargeoffset - 24*60 < 3*24*60 THEN 2 -- 2 days to 3 days
        WHEN unitdischargeoffset - 24*60 >= 3*24*60 AND unitdischargeoffset - 24*60 < 4*24*60 THEN 3 -- 3 days to 4 days
        WHEN unitdischargeoffset - 24*60 >= 4*24*60 AND unitdischargeoffset - 24*60 < 5*24*60 THEN 4 -- 4 days to 5 days
        WHEN unitdischargeoffset - 24*60 >= 5*24*60 AND unitdischargeoffset - 24*60 < 6*24*60 THEN 5 -- 5 days to 6 days
        WHEN unitdischargeoffset - 24*60 >= 6*24*60 AND unitdischargeoffset - 24*60 < 7*24*60 THEN 6 -- 6 days to 7 days
        WHEN unitdischargeoffset - 24*60 >= 7*24*60 AND unitdischargeoffset - 24*60 < 8*24*60 THEN 7 -- 7 days to 8 days
        WHEN unitdischargeoffset - 24*60 >= 8*24*60 AND unitdischargeoffset - 24*60 < 14*24*60 THEN 8 -- 1 week to 2 weeks
        WHEN unitdischargeoffset - 24*60 >= 14*24*60 THEN 9 -- More than 2 weeks
        ELSE NULL
    END AS remaining_los
FROM patient
WHERE unitdischargeoffset >= 24*60
  AND patientUnitStayID in ({considered_stay_id_str})
"""

df = pd.read_sql_query(query,conn)
df.to_csv(os.path.join(export_dir, "label_los.csv"),index=False,sep=',')
df.info()

In [ ]:
# Admission Drug

query = f"""
SELECT 
    patientunitstayid,
    drugoffset, 
    TRIM(drugname) AS drugname,
    drugdosage,
    drugUnit
FROM 
    admissiondrug
WHERE 
    drugdosage <> 0
    AND drugoffset > 24 * 60
    AND patientUnitStayID in ({considered_stay_id_str})
"""

df = pd.read_sql_query(query,conn)
df.to_csv(os.path.join(export_dir, "label_medication.csv"),index=False,sep=',')
df.info()

## Extract graph building information

ICD code and drug information

In [ ]:
query = f"""
SELECT 
	patientUnitStayID AS stay_id, 
	SPLIT_PART(icd9code, ',', 1) AS icd9_code,
	diagnosispriority
FROM diagnosis
WHERE SPLIT_PART(icd9code, ',', 1) IS NOT NULL
	AND SPLIT_PART(icd9code, ',', 1) <> ''
	AND SPLIT_PART(icd9code, ',', 1) NOT LIKE 'V%' 
	AND SPLIT_PART(icd9code, ',', 1) NOT LIKE 'E%'
    AND patientUnitStayID in ({considered_stay_id_str})
"""

df = pd.read_sql_query(query,conn, dtype={'icd9_code': str})
df.to_csv(os.path.join(export_dir, "diag.csv"),index=False, sep=',')
df.info()

In [ ]:
query = f"""
SELECT patientunitstayid, drugname, drughiclseqno
FROM admissiondrug
WHERE drugdosage <> 0
    AND drugoffset <= 24 * 60
    AND patientUnitStayID in ({considered_stay_id_str})
"""

df = pd.read_sql_query(query,conn)
df.to_csv(os.path.join(export_dir, "drug.csv"),index=False, sep=',')
df.info()